In [7]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import json
import traceback

import torch

import numpy as np
import cv2

from ocularrigidity.pipeline_config import RegistrationConfig
from ocularrigidity.segmentation.utils import get_choroid_segmentation_model
from ocularrigidity.segmentation.utils import (get_model,get_registration_model)
from ocularrigidity.consts import PLEX_WIDTH, AXIAL_PIXEL_SIZE_MM, TRANVERSAL_PIXEL_SIZE_MM
from ocularrigidity.data.spectralis import analyze, load_video
from ocularrigidity.registration.fused import segment_and_register
from ocularrigidity.pipeline_config import REGISTRATION
from ocularrigidity.data.compression import cube_to_mp4
from ocularrigidity.data.io import save_mask


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:

def to_plex_grid(frames, axial_mm, lateral_mm, width=PLEX_WIDTH):
    """Resample B-scans to the PLEX pixel size the networks were trained on."""
    T, H, W = frames.shape
    new_h = int(round(H * axial_mm / AXIAL_PIXEL_SIZE_MM / 32)) * 32
    new_w = int(round(W * lateral_mm / TRANVERSAL_PIXEL_SIZE_MM))
    out = np.stack(
        [cv2.resize(f, (new_w, new_h), interpolation=cv2.INTER_CUBIC) for f in frames]
    )
    width = min(width, new_w // 32 * 32)
    x0 = (new_w - width) // 2
    out = np.ascontiguousarray(out[:, :, x0 : x0 + width])
    return out, H * axial_mm / new_h, W * lateral_mm / new_w

In [3]:
seg_model = get_model()
reg_model = get_registration_model("reg_upernet_convnext_t_bmlines").to("cuda")

c:\Users\transformer\anaconda3\envs\pyOR\Lib\site-packages\torchmetrics\utilities\prints.py:43: UserWarning: DiceScore metric currently defaults to `average=micro`, but will change to`average=macro` in the v1.9 release. If you've explicitly set this parameter, you can ignore this warning.
  warnings.warn(*args, **kwargs)
C:\Users\transformer\Documents\Nicolas\mes-codes\SANSORI\OcularRigidity\src\ocularrigidity\segmentation\utils.py:36: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:839.)
  if float(next(model.parameters()).abs().max()) == 0.0:


In [4]:
PATH_GENERAL = Path("E:/SANSORI")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Segmentation de la choroide : meme echelle que first_cc_registration.py.
SEG_SCALE_FACTOR = 2.0
SEG_BATCH_SIZE = 8

# OUTPUT_SUBDIR = DEFAULT_OUTPUT_SUBDIR
OVERWRITE = False  # re-traiter une condition deja recalee 

for path_astro in PATH_GENERAL.iterdir():
    if not path_astro.is_dir():
        continue
    for path_moment in path_astro.iterdir():
        if not path_moment.match("*rigidity"):
            continue
        for path_condi in path_moment.iterdir():
            if not path_condi.is_dir():
                continue
            print(path_condi)
            this_vid,t_stamps,df = load_video(path_condi/'RawImages', return_fundus=False)
            this_vid_plex_grid,scale_axial,scale_lat = to_plex_grid(this_vid,0.0039, 0.0116)
            result = segment_and_register(this_vid_plex_grid, seg_model, reg_model, config=REGISTRATION, device=DEVICE)

            break
        break
    break

E:\SANSORI\01_210713001\210713001before_rigidity\210713001before_rigidity_OD


In [9]:
out = path_condi / "RawImages" / "registered_upernet_notebook" / "reg_frames.mp4"
out.parent.mkdir(parents=True, exist_ok=True)
cube_to_mp4(result.registered_frames, out, fps=17.86, crf=25)
save_mask(result.registered_masks, out.parent/"masks.npz")
print(out)
print(out.parent/"masks.npz")

E:\SANSORI\01_210713001\210713001before_rigidity\210713001before_rigidity_OD\RawImages\registered_upernet_notebook\reg_frames.mp4
E:\SANSORI\01_210713001\210713001before_rigidity\210713001before_rigidity_OD\RawImages\registered_upernet_notebook\masks.npz
